# 02 Spark Processing and Analysis

This notebook reads the cleaned dataset produced by notebook 01, performs transformations and aggregations using PySpark, and saves the results for the MongoDB loading step.

**Input:** `../data/processed/orders_core.parquet`  
**Output:** `../data/processed/customer_spending.parquet`, `top_categories.parquet`, `monthly_trends.parquet`, `customer_segments.parquet`

## 1. Environment Setup

Set JAVA_HOME so PySpark can locate the Java runtime before starting the Spark session.


In [ ]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ["PATH"]

## 2. Start Spark Session

Initialize the Spark session used for all processing in this notebook.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Olist_Spark_Processing").getOrCreate()
print("Spark version:", spark.version)

## 3. Load Processed Dataset

Read the cleaned and joined dataset produced by the ingestion notebook.

In [3]:
df = spark.read.parquet("../data/processed/orders_core.parquet")
df.printSchema()
print("Total rows:", df.count())

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- payment_value_total: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- total_order_value: double (nullable = true)

Total rows: 113425


## 4. Total Spending Per Customer

Calculate how much each unique customer has spent across all their orders. Payment values are deduplicated at order level before aggregating to avoid double counting from the order-item level structure.

In [4]:
from pyspark.sql.functions import col, sum, count, round

# deduplicate payment_value_total per order first to avoid double counting
from pyspark.sql.functions import first

spending_per_customer = (
    df.dropna(subset=["price"])
    .groupBy("customer_unique_id", "order_id")
    .agg(first("payment_value_total").alias("order_payment"))
    .groupBy("customer_unique_id")
    .agg(
        round(sum("order_payment"), 2).alias("total_spent"),
        count("order_id").alias("total_orders")
    )
    .orderBy(col("total_spent").desc())
)

spending_per_customer.show(10)
print("Unique customers:", spending_per_customer.count())

+--------------------+-----------+------------+
|  customer_unique_id|total_spent|total_orders|
+--------------------+-----------+------------+
|0a0a92112bd4c708c...|   13664.08|           1|
|da122df9eeddfedc1...|    7571.63|           2|
|763c8b1c9c68a0229...|    7274.88|           1|
|dc4802a71eae9be1d...|    6929.31|           1|
|459bef486812aa252...|    6922.21|           1|
|ff4159b92c40ebe40...|    6726.66|           1|
|4007669dec559734d...|    6081.54|           1|
|5d0a2980b292d0490...|    4809.44|           1|
|eebb5dda148d3893c...|    4764.34|           1|
|48e1ac109decbb877...|    4681.78|           1|
+--------------------+-----------+------------+
only showing top 10 rows
Unique customers: 95420


## 5. Top Product Categories by Revenue

Identify which product categories generate the most revenue and have the highest sales volume.

In [5]:
top_categories = (
    df.dropna(subset=["price"])
    .groupBy("product_category_name")
    .agg(
        round(sum("price"), 2).alias("total_revenue"),
        count("order_item_id").alias("total_items_sold")
    )
    .orderBy(col("total_revenue").desc())
)

top_categories.show(15)

+---------------------+-------------+----------------+
|product_category_name|total_revenue|total_items_sold|
+---------------------+-------------+----------------+
|         beleza_saude|   1258681.34|            9670|
|   relogios_presentes|   1205005.68|            5991|
|      cama_mesa_banho|   1036988.68|           11115|
|        esporte_lazer|    988048.97|            8641|
| informatica_acess...|    911954.32|            7827|
|     moveis_decoracao|    729762.49|            8334|
|           cool_stuff|    635290.85|            3796|
| utilidades_domest...|    632248.66|            6964|
|           automotivo|    592720.11|            4235|
|   ferramentas_jardim|    485256.46|            4347|
|           brinquedos|     483946.6|            4117|
|                bebes|    411764.89|            3065|
|           perfumaria|    399124.87|            3419|
|            telefonia|    323667.53|            4545|
|    moveis_escritorio|     273960.7|            1691|
+---------

## 6. Monthly Order Trends

Analyse how order volume and revenue change month by month across 2016 to 2018.

In [6]:
from pyspark.sql.functions import concat, lit, lpad

monthly_trends = (
    df.dropna(subset=["price"])
    .select("order_id", "year", "month", "payment_value_total")
    .dropDuplicates(["order_id"])
    .groupBy("year", "month")
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("payment_value_total"), 2).alias("monthly_revenue")
    )
    .orderBy("year", "month")
)

monthly_trends.show(30)

+----+-----+------------+---------------+
|year|month|total_orders|monthly_revenue|
+----+-----+------------+---------------+
|2016|    9|           3|         211.29|
|2016|   10|         308|       56884.89|
|2016|   12|           1|          19.62|
|2017|    1|         789|      137251.79|
|2017|    2|        1733|      286340.87|
|2017|    3|        2641|      432087.03|
|2017|    4|        2391|      412562.01|
|2017|    5|        3660|      586406.28|
|2017|    6|        3217|      503138.27|
|2017|    7|        3969|      585076.47|
|2017|    8|        4293|      668372.85|
|2017|    9|        4243|      720516.53|
|2017|   10|        4568|      769341.62|
|2017|   11|        7451|     1179307.63|
|2017|   12|        5624|       863675.0|
|2018|    1|        7220|     1108021.29|
|2018|    2|        6694|      987251.43|
|2018|    3|        7188|     1155206.57|
|2018|    4|        6934|     1159753.06|
|2018|    5|        6853|     1149843.99|
|2018|    6|        6160|      102

## 7. Customer Segmentation

Segment customers into High, Medium, and Low spenders based on their total spending to understand the distribution of customer value.

In [7]:
from pyspark.sql.functions import when

segmented = spending_per_customer.withColumn(
    "segment",
    when(col("total_spent") >= 500, "High Spender")
    .when(col("total_spent") >= 100, "Medium Spender")
    .otherwise("Low Spender")
)

segment_summary = (
    segmented.groupBy("segment")
    .agg(
        count("customer_unique_id").alias("customer_count"),
        round(sum("total_spent"), 2).alias("segment_revenue")
    )
    .orderBy(col("segment_revenue").desc())
)

segment_summary.show()

+--------------+--------------+---------------+
|       segment|customer_count|segment_revenue|
+--------------+--------------+---------------+
|Medium Spender|         46878|     9087410.36|
|  High Spender|          4425|     4095416.16|
|   Low Spender|         44117|     2663453.65|
+--------------+--------------+---------------+



## 8. Save Results for MongoDB Loading

Write all aggregated results to parquet files in the processed folder for next notebook to load into MongoDB.

In [8]:
spending_per_customer.write.mode("overwrite").parquet("../data/processed/customer_spending.parquet")
top_categories.write.mode("overwrite").parquet("../data/processed/top_categories.parquet")
monthly_trends.write.mode("overwrite").parquet("../data/processed/monthly_trends.parquet")
segmented.write.mode("overwrite").parquet("../data/processed/customer_segments.parquet")

print("All outputs saved successfully")

All outputs saved successfully
